# 05. 어느 센서를 언제 학습에서 잠시 빼야 할까?

**원자료를 지우지 않고, 안정된 운전 상태를 배우는 비교 실험에서 잠시 보류할 센서·시각을 표시한다. 고장 정답표가 아니다.** LOT는 연속해서 관측된 생산 묶음이며, 시각은 모두 2024년이다.

- 240038의 높은 `OP-Z4`는 원인 미확인으로 유지하고, 포함·제외 비교만 별도로 제안한다.
- 240058은 CP 저하와 온도 감소의 시각을 분리한다. 가열 구역뿐 아니라 과시효 온도·출력도 살핀다.
- 240061은 초기 온도 상승을 확인한 구간만 보류해 과거의 과도한 218분 표시를 줄인다.
- 시험 자료는 표시만 하며 제외하거나 기준 선택에 이용하지 않는다.

읽는 순서는 **관측 수 확인 → 제외 근거와 선택값 → 실제 적용 목록 → 판단 보류 목록 → 남은 자료 수**다. ‘제외’는 원본 행 삭제가 아니라 해당 센서를 예측하는 모델의 학습 또는 검증 기준 계산에서 쓰지 않는다는 뜻이다.

학습 9·검증 2·시험 3 LOT는 **기존 해석 실험의 잠정 분할**을 유지한 것이며 팀의 최종 합의 분할을 뜻하지 않는다. 학습에서는 모델 학습, 검증에서는 이상 판단 기준 계산에 해당 센서의 표시를 적용한다. 시험에도 공통 관측 수 조건은 적용하지만 사건 구간을 추가로 제외하지 않는다.

새 기본안으로 모델을 다시 학습한 것은 아니다. 파일 연결과 기존 성능 해석 주의점은 마지막 ‘다음 모델 실험에 전달할 내용’에서 확인한다.

이 문서의 ‘기존 06’은 이번 변경에 포함하지 않은 이전 참고 실험이다. **05 실행과 아래 행수 계산에는 06 파일이나 그 산출물이 필요하지 않다.**

원본 온도·조업이벤트 CSV는 저장소의 `data/` 또는 `data/raw/` 중 같은 폴더에 둔다. 계산 CSV는 항상 `data/processed/`, 그림은 `figures/05_exclusion_windows_namelesspark/`에 저장한다.


In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

DATA_DIR = os.path.join("..", "data")          # notebooks 폴더 기준 한 단계 위의 data 폴더
FIG_DIR = os.path.join("..", "figures")        # 그래프 저장 위치
# 저장소 루트에서 실행할 때도 같은 data와 figures 폴더를 사용합니다.
if os.path.isdir("data") and os.path.isdir("notebooks"):
    DATA_DIR = "data"
    FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

# 원자료는 data/ 또는 data/raw/에 함께 둡니다. 계산 결과는 data/processed에 저장합니다.
DATA_ROOT = DATA_DIR
if not os.path.isfile(os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")):
    RAW_DIR = os.path.join(DATA_DIR, "raw")
    if os.path.isfile(os.path.join(RAW_DIR, "T-CR1-CAL01_온도.csv")):
        DATA_DIR = RAW_DIR

# 기존 3개 (T- 센서 계측, G-02 사건 기록, G-01 기준 정보)
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")          # 소둔로 온도·제어출력
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")           # 강종변경 등, PLANT_CD 단위
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")       # 계획 정지, PLANT_CD 단위

# 2026-09-28 추가된 4개 (상위 이슈 #43)
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")             # 설비 사전, 키 EQP_TAG
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")            # 부품 사전, 키 PART_CD
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")             # 정비·고장 기록, 키 WO_NO + EQP_TAG
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")        # 부품 교체, 키 WO_NO + PART_CD

# 2026-10-08 추가된 용접 데이터
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")          # 용접 계측 데이터

ENC = "utf-8-sig"                              # G- 로 시작하는 마스터 CSV는 BOM 있음

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# 그래프 한글 깨짐 방지 - 설치된 폰트 중 있는 것을 골라 rcParams에 넣습니다
installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False         # 음수 기호가 □ 로 나오는 것 방지
rcParams["figure.dpi"] = 110


from pathlib import Path
import matplotlib.dates as mdates
from IPython.display import display

# 계산용 CSV는 data/processed, 새 그림은 이 노트북 전용 폴더에 저장합니다.
PROCESSED = Path(DATA_ROOT) / "processed"
PROCESSED.mkdir(exist_ok=True)
FIGURES = Path(FIG_DIR) / "05_exclusion_windows_namelesspark"
FIGURES.mkdir(exist_ok=True)

GROUPS = {
    "세정": ["CLN-IN"],
    "건조": ["TC-Z1", "TC-Z2", "OP-Z1", "OP-Z2"],
    "가열·균열": [f"TC-Z{i}" for i in range(3, 7)] + [f"OP-Z{i}" for i in range(3, 7)] + ["CP-Z4"],
    "과시효": ["TC-Z7", "TC-Z8", "TC-Z9", "OP-Z7", "OP-Z8"],
    "냉각": ["TC-OUT1", "TC-OUT2"],
}
CHANNELS = [sensor for sensors in GROUPS.values() for sensor in sensors]
HEAT_TC = [f"TC-Z{i}" for i in range(3, 7)]
HEAT_OP = [f"OP-Z{i}" for i in range(3, 7)]
HEAT_EIGHT = HEAT_TC + HEAT_OP
AGING_TC = ["TC-Z7", "TC-Z8"]
AGING_OP = ["OP-Z7", "OP-Z8"]
PHASE_GROUPS = {
    "가열·균열": (HEAT_TC, HEAT_EIGHT),
    "과시효": (AGING_TC, AGING_TC + AGING_OP),
}
TRAIN = [240001, 240024, 240037, 240038, 240044, 240058, 240061, 240069, 240091]
VALID = [240108, 240114]
TEST = [240117, 240121, 240133]
SPLIT = {}
for name, lots in [("train", TRAIN), ("valid", VALID), ("test", TEST)]:
    for lot in lots:
        SPLIT[lot] = name
SPLIT_NAME = {
    "train": "학습",
    "valid": "검증 기준 계산",
    "test": "시험: 제외하지 않음",
}
MIN_COUNT = 30
CP_CENTER = 0.45
CP_REVIEW_GAP = 0.03
CP_HOLD_GAP = 0.05

## 1. 1분 평균을 만들기 전에 관측 수를 확인한다

`30개 이상`은 공식 수집 사양이나 ‘30초 관측’이라는 뜻이 아니다. 이전 06과 비교할 수 있도록 유지한 임시 품질 조건이다. 2초 간격이면 대략 30개, 3초면 20개, 5초면 12개가 한 분에 들어오므로 **같은 조건을 다른 수집 주기의 자료에 그대로 적용하면 안 된다.** 아래에서 실제 간격과 탈락한 분을 확인한다.

한 분에 전체 행이 30개 있어도 특정 센서가 비어 있을 수 있다. 따라서 **센서마다 유효 관측 수를 세어 30개 미만인 평균만 비운다.** 시간 행은 남겨 ‘1분 전’과 ‘5분 전’ 간격을 보존하고, LOT 사이의 빈 날짜를 채우지 않는다. `CP-Z4M`은 값이 정확히 0이어서가 아니라, 정의가 불명확하고 `TC-Z5`와 거의 중복되는 별도 컬럼이어서 이번 분석에서 제외한다.

In [ ]:
raw = pd.read_csv(TEMP_CSV, encoding=ENC, parse_dates=["MEAS_DT"])
events = pd.read_csv(EVENT_CSV, encoding=ENC, parse_dates=["EVT_DT"])
raw = raw.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)
if raw.duplicated(["LOT_NO", "MEAS_DT"]).any():
    raise ValueError("같은 LOT·시각이 중복되어 평균에 두 번 반영될 수 있습니다.")

lots = raw.groupby("LOT_NO").agg(
    START=("MEAS_DT", "min"),
    END=("MEAS_DT", "max"),
    RAW_ROWS=("MEAS_DT", "size"),
)
grade_rows = events.PLANT_CD.eq("CR1") & events.EVT_TYPE.eq("강종변경")
grades = events.loc[grade_rows, ["EVT_DT", "AFT_VAL"]]
# 강종이 중복 연결되면 한 LOT의 행이 늘어나므로 일대일 연결을 확인합니다.
lots = lots.reset_index().merge(
    grades,
    left_on="START",
    right_on="EVT_DT",
    how="left",
    validate="one_to_one",
)
lots = lots.rename(columns={"AFT_VAL": "GRADE"}).set_index("LOT_NO")
if lots.GRADE.isna().any():
    raise ValueError("강종변경 기록과 연결되지 않은 LOT가 있습니다.")
lots["SPLIT"] = lots.index.map(SPLIT)

minute_parts = []
count_parts = []
gap_parts = []
for lot, group in raw.groupby("LOT_NO", sort=True):
    series = group.set_index("MEAS_DT")
    means = series[CHANNELS].resample("1min").mean()
    counts = series[CHANNELS].resample("1min").count()
    row_counts = series.resample("1min").size()
    means = means.where(counts.ge(MIN_COUNT))
    means["ROW_COUNT"] = row_counts
    means["LOT_NO"] = lot
    means["GRADE"] = lots.loc[lot, "GRADE"]
    means["SPLIT"] = SPLIT[lot]
    means["ELAPSED_MIN"] = (means.index - lots.loc[lot, "START"]).total_seconds() / 60
    counts["LOT_NO"] = lot
    minute_parts.append(means.reset_index())
    count_parts.append(counts.reset_index())
    gaps = series.index.to_series().diff().dt.total_seconds().dropna()
    for seconds, count in gaps.value_counts().sort_index().items():
        gap_parts.append({
            "LOT": lot,
            "자료용도": SPLIT_NAME[SPLIT[lot]],
            "관측간격_초": seconds,
            "횟수": count,
        })

minute = pd.concat(minute_parts, ignore_index=True)
counts = pd.concat(count_parts, ignore_index=True)
minute["QUALITY_OK"] = minute.ROW_COUNT.ge(MIN_COUNT)
minute["HEAT_TC_MEAN"] = minute[HEAT_TC].mean(axis=1, skipna=False)
gap_table = pd.DataFrame(gap_parts)

def lot_minutes(lot):
    return minute.loc[minute.LOT_NO.eq(lot)].set_index("MEAS_DT")

def save_csv(frame, name):
    frame.to_csv(PROCESSED / ("05_" + name + ".csv"), index=False, encoding=ENC)

quality_rows = []
for lot, group in minute.groupby("LOT_NO"):
    sensor_counts = counts.loc[group.index, CHANNELS]
    extra_bad = sensor_counts.lt(MIN_COUNT) & group.QUALITY_OK.to_numpy()[:, None]
    quality_rows.append({
        "LOT": lot,
        "강종": lots.loc[lot, "GRADE"],
        "자료용도": SPLIT_NAME[SPLIT[lot]],
        "전체_분": len(group),
        "관측30개이상_분": int(group.QUALITY_OK.sum()),
        "관측30개미만_분": int((~group.QUALITY_OK).sum()),
        "센서결측으로추가보류_센서분": int(extra_bad.to_numpy().sum()),
    })
quality = pd.DataFrame(quality_rows)
display(quality)
display(gap_table.pivot_table(index="LOT", columns="관측간격_초", values="횟수", fill_value=0))

quality_sensitivity = []
for minimum in [12, 20, 30, 45]:
    for split in ["train", "valid", "test"]:
        selected = minute.SPLIT.eq(split)
        usable = counts.loc[selected, CHANNELS].ge(minimum)
        quality_sensitivity.append({
            "분당최소관측수": minimum,
            "자료용도": SPLIT_NAME[split],
            "남는센서분": int(usable.to_numpy().sum()),
        })
display(pd.DataFrame(quality_sensitivity))
save_csv(quality, "sampling_quality")
save_csv(gap_table, "sampling_intervals")
save_csv(pd.DataFrame(quality_sensitivity), "sampling_sensitivity")

## 2. 구간의 뜻을 센서별로 정한다

이번 비교에서 먼저 보류하기로 선택한 현상은 **구역 온도의 공통 상승·하강과 정한 범위 밖의 CP 이탈**이다. 고장 확정 여부만으로 보류와 유지를 나눈 것은 아니다. 출력 왕복이나 반복 온도 파형은 별도의 적용 경계를 아직 정하지 않아 검토 상태로 남겼으며, 안정 상태나 고장 없음으로 판정한 것이 아니다.

| 관측 | 이번 기본안 | 뜻과 한계 |
| --- | --- | --- |
| 240038의 높은 `OP-Z4` | 강제 제외를 철회하고 비교 선택지로만 보관 | 같은 설비의 정비 기록은 있지만 고장모드가 비어 있고 비고는 ‘확인 중’. 강종·설정 차이와 설비 문제를 구분하지 못함 |
| 240058 후반 | CP 저하와 구역별 온도 감소의 시작을 따로 표시 | CP의 지속 저하가 먼저라는 이유로 다른 센서를 같은 시각부터 제외하지 않음 |
| 240061 초반 | 가열 8개와 과시효 `TC-Z7/Z8`·`OP-Z7/Z8` 4개를 구역별 안정 시각 전까지 보류 | CP는 약 0.45로 유지되어 함께 제외하지 않음. 주기 변화가 큰 `TC-Z9`는 자동으로 묶지 않음 |
| 짧은 CP 이탈 | `0.45 ± 0.05` 밖의 원자료가 들어 있는 1분 칸에서 CP만 보류 | 안정 상태를 배우는 실험의 임시 선택값. 고장 한계가 아니며 양·음 방향에 같은 기준 적용. 앞뒤 5분·다른 센서로 자동 확대하지 않음 |
| 240069 초기 `CLN-IN` | 검토 목록으로 이동 | 초기 상승은 확인되지만 실제 설정값·정상 상태가 없어 과거 313분 전체 제외를 확정하지 않음 |
| 240114 후반 | 가열 8개와 과시효 4개의 하강 시각을 따로 정해 검증 기준 계산에서 보류 | CP는 안정되어 보류하지 않음. 검증 LOT를 학습에 옮기지 않음 |

CP 기준의 **0.45는 기존 관찰 메모의 대표 수준이며 학습 자료 중앙값으로 대조한다.** 허용 편차 0.05는 이번 비교를 위한 선택이다. 0.03·0.05·0.10에서 얼마나 달라지는지 학습 자료만으로 비교하고, 검증·시험은 정한 기준을 그대로 적용해 관찰한다. 음의 짧은 저하를 양의 급상승과 다르게 취급하지 않는다.

240058·240114의 후반 경계는 **구역별 온도 평균이 해당 LOT 중앙값보다 3℃ 낮아진 뒤 끝까지 유지되는 시각**이다. 3℃는 운전 단계를 나누는 임시 선택값이다. LOT 전체와 이후 값을 사용하므로 실시간 경보 기준이 아닌 사후 표시다. 출력은 같은 순간부터 모두 감소하는 것이 아니라 상승·유지되다가 말기에 감소할 수도 있다. 온도와 출력의 관계가 달라지는 구역 전체의 운전 단계를 보류하는 선택이다.

가열 8개·과시효 4개의 공동 표시는 **이 노트북에서 가정한 구역의 온도·출력을 같은 운전 단계로 취급하는 임시 선택**이다. 출력 각각의 변화 시작이나 고장이 확인됐다는 뜻은 아니다. 반면 `TC-Z9`·냉각 온도는 자체 반복 파형과 후반 수준 변화를 분리한 경계를 아직 정하지 못했으므로 같은 시각을 자동 적용하지 않는다. 이 차이는 보류 규칙을 정한 범위의 차이이며, 남은 센서가 더 안정적이라는 결론이 아니다.

In [ ]:
applied_rows = []
review_rows = []
diagnostic_rows = []

def add_window(destination, lot, start, end, channels, reason, evidence):
    destination.append({
        "LOT": lot,
        "강종": lots.loc[lot, "GRADE"],
        "분할": SPLIT[lot],
        "시작": max(pd.Timestamp(start), lots.loc[lot, "START"]),
        "끝": min(pd.Timestamp(end), lots.loc[lot, "END"]),
        "채널": ";".join(channels),
        "사유": reason,
        "근거": evidence,
    })

def first_persistent(condition):
    # 마지막까지 조건이 유지되는 시각을 찾는 사후 분석입니다.
    kept = condition.fillna(False).iloc[::-1].cummin().iloc[::-1]
    if kept.any():
        return kept.loc[kept].index[0]
    return pd.NaT

cooling_starts = {}
for lot in [240058, 240114]:
    group = lot_minutes(lot)
    for zone, (temperatures, targets) in PHASE_GROUPS.items():
        temperature = group[temperatures].mean(axis=1, skipna=False).dropna()
        reference = temperature.median()
        start = first_persistent(temperature.le(reference - 3))
        cooling_starts[(lot, zone)] = start
        add_window(
            applied_rows, lot, start, lots.loc[lot, "END"], targets,
            f"{zone} 후반 온도 감소와 출력 변화",
            f"해당 구역 온도 평균이 LOT 중앙값 {reference:.3f}보다 3℃ 낮은 상태로 끝까지 유지. "
            "온도가 내려가며 출력과의 관계가 달라지는 운전 단계에 적용. 출력은 상승·유지되다가 말기에 감소할 수도 있음. 각 센서의 최초 변화 시각이라는 뜻은 아님.",
        )
        diagnostic_rows.append({
            "LOT": lot,
            "판단": f"{zone} 온도 감소와 출력 변화",
            "기준수준": reference,
            "구간시작": start,
            "사후분석": True,
        })

cp_058 = lot_minutes(240058)["CP-Z4"].dropna()
cp_decline_start = first_persistent(cp_058.lt(0.40))
add_window(
    applied_rows, 240058, cp_decline_start, lots.loc[240058, "END"], ["CP-Z4"],
    "CP의 낮은 수준이 마지막까지 유지",
    "유효 1분 평균이 0.40 미만으로 끝까지 지속. 온도·출력의 하강 시각과 분리한 사후 표시.",
)
diagnostic_rows.append({
    "LOT": 240058,
    "판단": "CP 지속 저하",
    "기준수준": 0.40,
    "구간시작": cp_decline_start,
    "사후분석": True,
})
display(pd.DataFrame(diagnostic_rows).sort_values(["LOT", "구간시작"]))

## 3. 240061: 초기 상승 구간을 너무 넓게 보류하지 않는다

기존의 ‘기울기 조건이 마지막까지 깨지지 않는 시각’은 218분을 표시했다. 중간의 온도 흔들림 하나 때문에 종료 시각이 뒤로 밀리므로 폐기한다.

이번에는 **마지막 30분 안의 유효 구역 온도 평균의 중앙값**을 비교 수준으로 놓는다. 가열 구역은 `TC-Z3~Z6` 4개, 과시효 구역은 `TC-Z7/Z8` 2개의 평균을 각각 사용한다. 그 수준의 **±2℃ 안에 20분 연속 머무는 첫 구간의 시작**부터 자료를 남긴다. 1·2·3℃와 10·20·30분을 비교한다. 값은 물리적 정상 한계가 아니라 구간을 나누는 선택값이며, 기준 수준과 연속 유지 여부 모두 이후 자료를 사용하는 **사후 주석**이다.

‘안정 구간 시작’과 ‘20분이 지났음을 확인할 수 있는 시각’은 다르다. 실시간 판별기로 사용할 수 없다. 초기·후반 출력 수준도 비교해 온도만 보류할지 판단한다. CP가 안정적이라는 이유만으로 나머지 모든 센서가 정상이라고 판단하지는 않는다.

가열 구역은 **02:30분까지 보류하고 02:31분부터 남긴다.** 02:31~02:50의 20개 분 평균이 완성되는 02:51에 연속 조건 충족을 확인할 수 있지만, 확인 시각까지 추가로 보류하지 않는다. 여기서 ‘안정 구간 시작’은 이번 조건을 처음 만족한 시각이라는 뜻이며, 03:12 이후의 출력 0과 온도 왕복처럼 뒤의 모든 센서·시각이 계속 안정적이라는 뜻은 아니다.

In [ ]:
startup = lot_minutes(240061)
startup_rows = []
for zone, (temperatures, targets) in PHASE_GROUPS.items():
    temperature = startup[temperatures].mean(axis=1, skipna=False)
    reference = temperature.tail(30).median()
    for tolerance in [1, 2, 3]:
        near = temperature.sub(reference).abs().le(tolerance)
        for duration in [10, 20, 30]:
            confirmed = near.rolling(duration, min_periods=duration).sum().eq(duration)
            confirmation_bin = confirmed.loc[confirmed].index[0]
            start = confirmation_bin - pd.Timedelta(minutes=duration - 1)
            startup_rows.append({
                "구역": zone,
                "허용폭_℃": tolerance,
                "연속확인_분": duration,
                "비교수준_℃": reference,
                "안정구간시작": start,
                "확인가능시각": confirmation_bin + pd.Timedelta(minutes=1),
                "초기보류_분": int((start - lots.loc[240061, "START"]).total_seconds() / 60),
            })
startup_sensitivity = pd.DataFrame(startup_rows)
chosen_rows = startup_sensitivity.loc[
    startup_sensitivity["허용폭_℃"].eq(2) & startup_sensitivity["연속확인_분"].eq(20)
]
for _, chosen in chosen_rows.iterrows():
    temperatures, targets = PHASE_GROUPS[chosen["구역"]]
    start = chosen["안정구간시작"]
    add_window(
        applied_rows, 240061, lots.loc[240061, "START"], start - pd.Timedelta(seconds=1),
        targets, f"{chosen['구역']} 초기 온도·출력의 운전 단계 차이",
        f"후반 30분 비교수준 {chosen['비교수준_℃']:.3f}℃의 ±2℃에 20분 연속 머문 구간 시작 {start}. "
        "출력도 초반·후반 수준 차이가 있어 함께 적용. CP·TC-Z9는 포함하지 않음. 사후 선택값.",
    )
heat_chosen = chosen_rows.loc[chosen_rows["구역"].eq("가열·균열")].iloc[0]
startup_reference = heat_chosen["비교수준_℃"]
stable_start = heat_chosen["안정구간시작"]
comparison_sensors = HEAT_TC + HEAT_OP + AGING_TC + AGING_OP + ["CP-Z4"]
startup_comparison = pd.DataFrame({
    "초기15분_중앙값": startup[comparison_sensors].head(15).median(),
    "마지막60분_중앙값": startup[comparison_sensors].tail(60).median(),
})
display(startup_sensitivity)
display(startup_comparison.round(5))
save_csv(startup_sensitivity, "startup_sensitivity")
save_csv(startup_comparison.rename_axis("센서").reset_index(), "startup_sensor_comparison")

## 4. CP는 양쪽 방향을 같은 기준으로 찾는다

**0.45와의 차이가 0.05보다 큰 원자료**를 찾고, 그러한 값이 있는 분만 CP 보류 대상으로 표시한다. 시간상 연속된 분은 한 구간으로 묶는다. 표의 시작·끝은 해당 구간에서 기준을 벗어난 첫·마지막 원자료 시각이다. 따라서 1초의 이탈이 있더라도 모델의 분석 단위가 1분이면 해당 분 전체가 표시된다.

0.03보다 크지만 0.05 이내인 이탈은 검토 목록에만 넣는다. 240024의 음의 저하, 240038의 짧은 양의 상승에도 같은 기준을 쓴다. 240108과 시험 LOT의 하강도 같은 방식으로 관찰하지만, 그것을 잘 잡도록 선택값을 조정하지 않는다. 240058의 지속 저하는 앞에서 별도로 표시했으므로 이후의 CP 구간을 중복으로 더하지 않는다.

240058에서 원자료 기준 이탈은 07:33:21에 시작하고, **1분 평균이 0.40 미만으로 끝까지 유지되는 조건은 07:43에 시작**한다. 서로 다른 두 조건이므로 앞의 짧은 이탈과 뒤의 지속 저하를 나누어 적는다. 실제 CP 보류 표시는 07:33부터 이어진다.

In [ ]:
def cp_events(frame, gap):
    hits = frame.loc[frame["CP-Z4"].sub(CP_CENTER).abs().gt(gap)].copy()
    hits["MINUTE"] = hits.MEAS_DT.dt.floor("min")
    rows = []
    for lot, group in hits.groupby("LOT_NO", sort=True):
        marked_minutes = group.MINUTE.drop_duplicates().sort_values()
        groups = marked_minutes.diff().gt(pd.Timedelta(minutes=1)).cumsum()
        for _, times in marked_minutes.groupby(groups):
            selected = group.loc[group.MINUTE.isin(times)]
            rows.append({
                "LOT": lot,
                "강종": lots.loc[lot, "GRADE"],
                "분할": SPLIT[lot],
                "시작": selected.MEAS_DT.min(),
                "끝": selected.MEAS_DT.max(),
                "이탈분": len(times),
                "이탈관측수": len(selected),
                "CP최소": selected["CP-Z4"].min(),
                "CP최대": selected["CP-Z4"].max(),
                "편차기준": gap,
            })
    return pd.DataFrame(rows)

training = raw.loc[raw.LOT_NO.isin(TRAIN)]
cp_training_center = training["CP-Z4"].median()
print(f"학습 자료 CP 중앙값: {cp_training_center:.6f}; 선택한 대표 수준: {CP_CENTER}")
cp_sensitivity_rows = []
for gap in [0.03, 0.05, 0.10]:
    table = cp_events(training, gap)
    for lot in TRAIN:
        group = table.loc[table.LOT.eq(lot)]
        cp_sensitivity_rows.append({
            "허용편차": gap,
            "LOT": lot,
            "이탈구간수": len(group),
            "이탈분": int(group["이탈분"].sum()),
            "이탈관측수": int(group["이탈관측수"].sum()),
        })
cp_sensitivity = pd.DataFrame(cp_sensitivity_rows)
cp_selected = cp_events(raw, CP_HOLD_GAP)
cp_review = cp_events(raw, CP_REVIEW_GAP)
display(cp_sensitivity)
display(cp_selected)
save_csv(cp_sensitivity, "cp_train_sensitivity")
save_csv(cp_selected, "cp_selected_events")
save_csv(cp_review, "cp_review_events")

for _, event in cp_selected.iterrows():
    start = event["시작"]
    end = event["끝"]
    if event.LOT == 240058:
        if start >= cp_decline_start:
            continue
        end = min(end, cp_decline_start - pd.Timedelta(seconds=1))
    destination = applied_rows
    purpose = "안정 상태 학습의 임시 보류이며 고장 정답이 아님."
    if event["분할"] == "test":
        destination = review_rows
        purpose = "시험 자료의 관찰 기록이며 학습·채점 제외에 적용하지 않음."
    interval_rows = raw.LOT_NO.eq(event.LOT) & raw.MEAS_DT.between(start, end)
    deviation_rows = raw["CP-Z4"].sub(CP_CENTER).abs().gt(CP_HOLD_GAP)
    interval_values = raw.loc[interval_rows & deviation_rows, "CP-Z4"]
    add_window(
        destination, int(event.LOT), start, end, ["CP-Z4"],
        "CP의 짧고 큰 이탈",
        f"0.45와 원자료 차이 >0.05. 이 구간의 이탈 관측 {len(interval_values)}개, 범위 {interval_values.min():.6f}~{interval_values.max():.6f}. "
        f"{purpose} 앞뒤 확대 없음.",
    )

# 큰 이탈과 겹친 사건 전체를 버리지 않고, 이미 표시한 분만 뺍니다.
already_marked = pd.Series(False, index=raw.index)
raw_minutes = raw.MEAS_DT.dt.floor("min")
for window in applied_rows + review_rows:
    if window["채널"] != "CP-Z4":
        continue
    marked = raw.LOT_NO.eq(window["LOT"])
    marked &= raw_minutes.between(window["시작"].floor("min"), window["끝"].floor("min"))
    already_marked |= marked
cp_review_only = cp_events(raw.loc[~already_marked], CP_REVIEW_GAP)
for _, event in cp_review_only.iterrows():
    add_window(
        review_rows, int(event.LOT), event["시작"], event["끝"], ["CP-Z4"],
        "작은 CP 이탈: 기본안에서는 유지",
        f"대표 수준과 차이가 0.03을 넘는 분 중 이미 표시한 큰 이탈·지속 저하의 분을 뺀 나머지. "
        f"원자료 범위 {event.CP최소:.6f}~{event.CP최대:.6f}. 검토 목적이며 기준 최적화에 쓰지 않음.",
    )

## 5. 적용 목록과 검토 목록을 분리한다

**적용 목록의 행은 모두 기본안에서 실제로 표시한 구간**이다. ‘학습 제외: 아니오’ 행을 섞지 않는다. 검토 목록의 구간은 기본안에 영향을 주지 않는다. 시험은 CP 이탈이나 초기 온도 상승이 있어도 검토 목록에만 놓는다.

**검토 목록의 시작·끝은 선택한 살펴볼 범위**이며 현상 전체의 시작·종료나 고장 지속시간을 뜻하지 않는다. 예를 들어 LOT 240061의 06/15 03:26:59와 LOT 240069의 07/08 05:05:59는 비교한 `TC-Z3`의 낮은 분까지 포함하려고 정했다. 출력 회복이나 현상 종료가 그 시각에 확인됐다는 뜻이 아니다.

240038의 `OP-Z4`는 정상이라고 확정한 것이 아니라, **고장 근거 없이 강제 제외하지 않기로 한 것**이다. 같은 학습·검증·시험 분할에서 이 LOT의 출력만 포함/제외하는 후속 비교 선택지를 별도 파일로 남긴다. 시험 결과를 보고 둘 중 유리한 안을 선택하면 안 된다.

In [ ]:
review_specs = [
    (240038, ["OP-Z4"], "높은 출력 수준의 원인 미확인", "정비 WO-2024-00627은 CAL01 일치, 고장모드 공란·비고 확인 중·부품교체 없음. 강종·설정·설비 영향 구분 불가."),
    (240061, ["TC-Z9"], "다른 온도 상승과 반복 파형의 구분", "TC-Z9는 큰 반복 파형이 있어 인접 온도·출력 네 센서의 초기 보류 시각을 자동으로 복사하지 않음."),
    (240058, ["TC-Z9"], "반복 파형과 후반 하강의 구분", "인접 온도·출력의 하강은 적용하되 TC-Z9의 큰 반복 파형과 운전 전환을 분리 검토. 같은 구역이라는 이유만으로 자동 확대하지 않음."),
    (240114, ["TC-Z9"], "반복 파형과 후반 하강의 구분", "TC-Z7/Z8·OP-Z7/Z8과 달리 큰 반복 파형이 있어 별도 검토. 검증 구간의 정상 여부가 확정된 것은 아님."),
    (240069, ["CLN-IN"], "낮게 시작한 세정조 온도의 점진적 상승", "과거 마지막 3시간과의 차이로 약 313분을 제외했던 선택을 철회. 실제 설정값·독립 센서가 없어 원인과 정상 수준 미확인."),
    (240091, ["TC-Z7", "TC-Z8", "TC-Z9"], "높은 기준 수준과 반복되는 온도 파형", "높은 값·반복만으로 제외하지 않음. 동일 강종과 다른 강종의 LOT 비교 및 설정값 확인 필요."),
]
for lot, sensors, reason, evidence in review_specs:
    add_window(review_rows, lot, lots.loc[lot, "START"], lots.loc[lot, "END"], sensors, reason, evidence)
add_window(
    review_rows, 240117, lots.loc[240117, "START"],
    lots.loc[240117, "START"] + pd.Timedelta(minutes=60) - pd.Timedelta(seconds=1),
    HEAT_EIGHT, "시험 LOT의 첫 60분 온도 상승 관찰",
    "첫 60분을 살펴보는 주석. 상승 종료 시각이나 제외 구간을 뜻하지 않음. 시험에서 비교 수준을 추정하지 않음.",
)
add_window(
    review_rows, 240133, "2024-12-28 15:54:00", lots.loc[240133, "END"], CHANNELS,
    "시험 LOT의 정비 감지 이후 관찰",
    "WO-2024-02105 감지 시각 이후. 교체 사유는 예방이며 측정 고장 정답은 아님. 시험 채점에서 유지.",
)

# 기본안 밖에 남는 큰 변화도 정상으로 확정하지 않고 검토 목록에 둡니다.
def longest_recorded_run(source, level):
    matching = source["OP-Z3"].eq(level)
    run_number = matching.ne(matching.shift()).cumsum()
    selected = source.loc[matching]
    runs = selected.groupby(run_number.loc[matching]).MEAS_DT.agg(["min", "max", "size"])
    return runs.loc[runs["size"].idxmax()]

output_reviews = [
    {
        "LOT": 240061,
        "시작": "2024-06-15 03:12:03",
        "끝": "2024-06-15 03:26:59",
        "온도최고분": "2024-06-15 03:15",
        "온도최저분": "2024-06-15 03:26",
        "사유": "초기 보류 이후 출력 0과 온도 왕복",
    },
    {
        "LOT": 240069,
        "시작": "2024-07-08 04:29:20",
        "끝": "2024-07-08 05:05:59",
        "온도최고분": "2024-07-08 04:53",
        "온도최저분": "2024-07-08 05:05",
        "사유": "출력 상단 평탄값 변화와 출력 0",
    },
    {
        "LOT": 240091,
        "시작": "2024-09-09 07:07:14",
        "끝": "2024-09-09 07:28:56",
        "온도최고분": "2024-09-09 07:10",
        "온도최저분": "2024-09-09 07:26",
        "사유": "출력 0 이후 높은 평탄값과 온도 왕복",
    },
]
additional_review_rows = []
for spec in output_reviews:
    lot = spec["LOT"]
    source = raw.loc[raw.LOT_NO.eq(lot)]
    selected = source.loc[source.MEAS_DT.between(spec["시작"], spec["끝"])]
    zero_run = longest_recorded_run(selected, 0)
    group = lot_minutes(lot)
    high = group.loc[pd.Timestamp(spec["온도최고분"]), "TC-Z3"]
    low = group.loc[pd.Timestamp(spec["온도최저분"]), "TC-Z3"]
    evidence = (
        f"연이어 기록된 OP-Z3=0: {zero_run['min']:%H:%M:%S}~{zero_run['max']:%H:%M:%S}, "
        f"{int(zero_run['size'])}개 관측. TC-Z3 유효 1분 평균은 "
        f"{pd.Timestamp(spec['온도최고분']):%H:%M} {high:.3f}℃에서 "
        f"{pd.Timestamp(spec['온도최저분']):%H:%M} {low:.3f}℃로 변화. "
    )
    if lot == 240061:
        evidence += "가열 구역 초기 보류 종료 02:31 이후에 발생. "
    if lot == 240069:
        before = source.loc[source.MEAS_DT.lt(pd.Timestamp(spec["시작"]))]
        evidence += f"04:29:20 이전 관측 최고 {before['OP-Z3'].max():.4f}%, 이후 검토 구간 최고 {selected['OP-Z3'].max():.4f}%. 실제 제어 상한 미확인. "
    if lot == 240091:
        upper = selected["OP-Z3"].max()
        upper_run = longest_recorded_run(selected, upper)
        evidence += f"이후 {upper:.4f}% 평탄 관측 {upper_run['min']:%H:%M:%S}~{upper_run['max']:%H:%M:%S}. 실제 제어 상한 미확인. "
    evidence += "출력 0은 결측으로 바꾸지 않는 실제 값. 유사한 왕복이 다른 구간에도 있어 제어 반응과 고장을 구분할 근거가 부족하므로 자동 제외하지 않음."
    add_window(
        additional_review_rows, lot, spec["시작"], spec["끝"], ["OP-Z3", "TC-Z3"],
        spec["사유"], evidence,
    )

tail_reviews = [
    (240058, "2024-06-09 08:09", "2024-06-09 08:50", "2024-06-09 08:02", "2024-06-09 08:45"),
    (240114, "2024-11-04 20:09", "2024-11-04 20:50", "2024-11-04 20:03", "2024-11-04 20:44"),
]
for lot, peak_before, peak_after, low_before, low_after in tail_reviews:
    group = lot_minutes(lot)
    start = cooling_starts[(lot, "가열·균열")]
    evidence = (
        f"유효 1분 평균 TC-OUT1 봉우리: {pd.Timestamp(peak_before):%H:%M} "
        f"{group.loc[peak_before, 'TC-OUT1']:.3f}℃ → {pd.Timestamp(peak_after):%H:%M} "
        f"{group.loc[peak_after, 'TC-OUT1']:.3f}℃. TC-OUT2 골: {pd.Timestamp(low_before):%H:%M} "
        f"{group.loc[low_before, 'TC-OUT2']:.3f}℃ → {pd.Timestamp(low_after):%H:%M} "
        f"{group.loc[low_after, 'TC-OUT2']:.3f}℃. "
    )
    if lot == 240058:
        previous = group.loc[group.index < start, "TC-Z9"]
        later = group.loc[group.index >= start, "TC-Z9"]
        evidence += (
            f"TC-Z9의 {start:%H:%M} 경계 전/후 최저 {previous.min():.3f}/{later.min():.3f}℃. "
            "앞부분 최저는 첫 관측 05:26의 값이라 골인지 알 수 없음. "
            "큰 봉우리와 골이 각각 한 번만 뚜렷하고 뒤쪽 반등은 작아 같은 반복 위치인지 확인할 수 없음. "
            "냉각은 상세 LOT 노트북에서 TC-OUT1의 봉우리 5개·골 5개(같은 종류끼리 41~43분), "
            "TC-OUT2의 봉우리 4개·골 5개(같은 종류끼리 41~44분)를 대조함. "
            "두 냉각 센서 모두 후반 봉우리·골이 앞선 같은 종류의 값들보다 낮으므로 단순한 반복 위치 차이만은 아님. "
            "세부 비교와 재계산은 07_lot_anomaly_review_namelesspark/06_LOT_240058.ipynb 참고. "
        )
    if lot == 240114:
        first = group.loc["2024-11-04 17:42", "TC-Z9"]
        last = group.loc["2024-11-04 20:58", "TC-Z9"]
        evidence += f"TC-Z9의 두 봉우리(16:47·20:03)에서 각각 55분 뒤는 {first:.3f}/{last:.3f}℃. 같은 경과시간이 같은 물리 조건을 보장하지는 않음. "
    evidence += f"검토 시작 {start:%H:%M}은 가열 온도의 기존 참고 경계이며 이 세 센서의 확정 변화 시작이 아님. "
    reason = "후반 반복 온도 파형의 수준 변화"
    if lot == 240058:
        reason = "후반 냉각 온도·TC-Z9 변화"
        evidence += (
            "후반 변화는 한 번 관측됐고 같은 DP590의 다른 LOT가 없어 원인과 유지 기간을 확인할 수 없음. "
            "새 보류 경계도 정하지 않았으므로 자동 제외하지 않음."
        )
    else:
        evidence += "반복 파형과 말기 운전 단계의 영향을 분리해야 하므로 자동 제외하지 않음."
    add_window(
        additional_review_rows, lot, start, lots.loc[lot, "END"], ["TC-Z9", "TC-OUT1", "TC-OUT2"],
        reason, evidence,
    )
# 건조 구역은 별도 보류 기준이 없으므로 관찰 근거만 추가합니다.
drying_reviews = [
    (240058, "2024-06-09 08:40:00", "2024-06-09 08:50", "2024-06-09 09:02"),
    (240114, "2024-11-04 20:30:00", "2024-11-04 20:46", "2024-11-04 20:58"),
]
drying_channels = GROUPS["건조"]
for lot, start, before, after in drying_reviews:
    start = pd.Timestamp(start)
    before = pd.Timestamp(before)
    after = pd.Timestamp(after)
    end = lots.loc[lot, "END"]
    source = raw.loc[raw.LOT_NO.eq(lot) & raw.MEAS_DT.between(start, end)]
    group = lot_minutes(lot)[drying_channels]
    missing = source[drying_channels].isna().sum()
    last_minute = source.loc[source.MEAS_DT.dt.floor("min").eq(end.floor("min"))]
    evidence = (
        f"선택한 검토 구간의 원자료 {len(source)}행, "
        f"네 센서 결측 합계 {int(missing.sum())}개. "
        f"마지막 {end:%H:%M}분 원자료는 {len(last_minute)}개. "
        f"센서별 관측 30개 이상인 1분 평균의 {before:%H:%M} → {after:%H:%M} 비교: "
    )
    for sensor in drying_channels:
        unit = "℃" if sensor.startswith("TC") else "%"
        evidence += f"{sensor} {group.loc[before, sensor]:.3f} → {group.loc[after, sensor]:.3f}{unit}. "
    if lot == 240058:
        evidence += (
            "이 비교 구간에서 두 출력과 TC-Z2는 매분 감소하지만 TC-Z1은 중간 반등이 있음. "
            "OP-Z1은 그보다 앞선 08:47~08:50에는 반등하므로 전체 후반의 단조 감소가 아님. "
        )
    if lot == 240114:
        evidence += (
            "20:59분은 관측 30개 미만이므로 1분 평균을 쓰지 않고 마지막 유효분 20:58과 비교. "
            "이 비교 구간에서 두 출력은 매분 감소하지만 TC-Z1은 오히려 오르고 TC-Z2는 하강 후 반등. "
            "OP-Z1은 그보다 앞선 20:37~20:46에는 반등하므로 전체 후반의 단조 감소가 아님. "
        )
    evidence += (
        "마지막 원자료에서 두 출력이 0이나 각 1개 관측뿐이므로 연속 상수는 아님. "
        f"{start:%H:%M}은 살펴볼 범위의 시작이며 현상 시작이나 정지 명령 시각이 아님. "
    )
    if lot == 240058:
        evidence += "같은 DP590의 다른 LOT가 없어 고장·건조 중단을 확정하지 않음. "
    if lot == 240114:
        evidence += "검증 자료의 관찰이며 고장·건조 중단을 확정하지 않음. 이 자료로 임계값이나 보류 규칙을 조정하지 않음. "
    evidence += "건조 구역의 별도 보류 기준을 정하지 않았으므로 기본안에서는 유지."
    add_window(
        additional_review_rows, lot, start, end, drying_channels,
        "후반 건조 온도·출력 변화: 별도 기준 미정으로 유지", evidence,
    )

review_rows.extend(additional_review_rows)

def finish_windows(rows, prefix):
    frame = pd.DataFrame(rows).sort_values(["LOT", "시작", "채널"]).reset_index(drop=True)
    frame.insert(0, "구간ID", [f"{prefix}{i + 1:03d}" for i in range(len(frame))])
    return frame

applied = finish_windows(applied_rows, "A")
review = finish_windows(review_rows, "R")
if applied.LOT.isin(TEST).any():
    raise ValueError("시험 LOT를 적용 제외 목록에 넣을 수 없습니다.")
optional = pd.DataFrame([{
    "선택지": "240038 OP-Z4만 제외하는 비교",
    "LOT": 240038,
    "시작": lots.loc[240038, "START"],
    "끝": lots.loc[240038, "END"],
    "채널": "OP-Z4",
    "기본안에적용": False,
    "이유": "원인 미확인. 포함·제외 민감도 비교용이며 현재 모델을 다시 실행한 결과는 아님.",
}])
save_csv(applied, "applied_windows")
save_csv(review, "review_windows")
save_csv(optional, "optional_comparison")
save_csv(pd.DataFrame(diagnostic_rows), "rule_diagnostics")

def show_windows(frame):
    view = frame[["구간ID", "LOT", "강종", "분할", "시작", "끝", "채널", "사유"]].copy()
    view["분할"] = view["분할"].map(SPLIT_NAME)
    view["채널"] = view["채널"].str.replace(";", " · ", regex=False)
    display(view)

print("실제로 적용한 센서별 구간")
show_windows(applied)
print("판단을 보류한 검토 구간: 기본안에서는 유지")
show_windows(review)
display(optional)

### 실행 전에도 확인할 수 있는 현재 적용 목록

아래 표는 현재 원자료에서 재계산한 결과다. 원자료나 선택값이 바뀌면 위 코드의 새 결과와 함께 갱신해야 한다. 검토 목록과 선택 비교안은 포함하지 않았다.

**가열 8개** = `TC-Z3`·`TC-Z4`·`TC-Z5`·`TC-Z6`·`OP-Z3`·`OP-Z4`·`OP-Z5`·`OP-Z6`. **과시효 4개** = `TC-Z7`·`TC-Z8`·`OP-Z7`·`OP-Z8`. `TC-Z9`는 두 묶음에 포함되지 않는다.

CP의 짧은 이탈은 원자료 첫·마지막 이탈 시각, 운전 단계는 사후 조건의 경계를 적었다. 모델과 그림의 보류 범위는 시작·끝이 들어 있는 **1분 칸 전체**다. 예를 들어 04:59:05~05:00:00은 04:59:00 이상 05:01:00 미만의 두 분을 표시한다.

| 구간 | LOT | 자료 용도 | 대상 센서 | 표시 구간(2024년) | 판단 |
| --- | --- | --- | --- | --- | --- |
| A001 | 240024 | 학습 | `CP-Z4` | 03/08 04:59:05 ~ 03/08 05:00:00 | CP의 짧고 큰 이탈 |
| A002 | 240037 | 학습 | `CP-Z4` | 04/13 23:20:26 ~ 04/13 23:21:00 | CP의 짧고 큰 이탈 |
| A003 | 240037 | 학습 | `CP-Z4` | 04/13 23:57:32 ~ 04/13 23:59:20 | CP의 짧고 큰 이탈 |
| A004 | 240037 | 학습 | `CP-Z4` | 04/14 03:10:55 ~ 04/14 03:11:19 | CP의 짧고 큰 이탈 |
| A005 | 240038 | 학습 | `CP-Z4` | 04/16 13:37:54 ~ 04/16 13:38:20 | CP의 짧고 큰 이탈 |
| A006 | 240058 | 학습 | `CP-Z4` | 06/09 07:33:21 ~ 06/09 07:42:59 | CP의 짧고 큰 이탈 |
| A007 | 240058 | 학습 | `CP-Z4` | 06/09 07:43:00 ~ 06/09 09:02:53 | CP의 낮은 수준이 마지막까지 유지 |
| A008 | 240058 | 학습 | 가열 8개 | 06/09 08:14:00 ~ 06/09 09:02:53 | 가열·균열 후반 온도 감소와 출력 변화 |
| A009 | 240058 | 학습 | 과시효 4개 | 06/09 08:21:00 ~ 06/09 09:02:53 | 과시효 후반 온도 감소와 출력 변화 |
| A010 | 240061 | 학습 | 가열 8개 | 06/15 01:04:00 ~ 06/15 02:30:59 | 가열·균열 초기 온도·출력의 운전 단계 차이 |
| A011 | 240061 | 학습 | 과시효 4개 | 06/15 01:04:00 ~ 06/15 02:14:59 | 과시효 초기 온도·출력의 운전 단계 차이 |
| A012 | 240069 | 학습 | `CP-Z4` | 07/07 18:59:57 ~ 07/07 19:02:12 | CP의 짧고 큰 이탈 |
| A013 | 240069 | 학습 | `CP-Z4` | 07/07 19:08:05 ~ 07/07 19:09:06 | CP의 짧고 큰 이탈 |
| A014 | 240069 | 학습 | `CP-Z4` | 07/07 19:11:18 ~ 07/07 19:11:55 | CP의 짧고 큰 이탈 |
| A015 | 240069 | 학습 | `CP-Z4` | 07/07 19:21:11 ~ 07/07 19:22:00 | CP의 짧고 큰 이탈 |
| A016 | 240069 | 학습 | `CP-Z4` | 07/07 19:38:12 ~ 07/07 19:39:25 | CP의 짧고 큰 이탈 |
| A017 | 240069 | 학습 | `CP-Z4` | 07/07 19:41:27 ~ 07/07 19:42:04 | CP의 짧고 큰 이탈 |
| A018 | 240069 | 학습 | `CP-Z4` | 07/08 02:30:46 ~ 07/08 02:38:39 | CP의 짧고 큰 이탈 |
| A019 | 240091 | 학습 | `CP-Z4` | 09/08 23:33:42 ~ 09/08 23:33:50 | CP의 짧고 큰 이탈 |
| A020 | 240114 | 검증 기준 계산 | 가열 8개 | 11/04 20:10:00 ~ 11/04 20:59:02 | 가열·균열 후반 온도 감소와 출력 변화 |
| A021 | 240114 | 검증 기준 계산 | 과시효 4개 | 11/04 20:19:00 ~ 11/04 20:59:02 | 과시효 후반 온도 감소와 출력 변화 |

### 기본안에서 남겼지만 추가 확인이 필요한 7개 구간

**아래 구간은 검토 목록에만 추가했다. 기본 적용 21구간과 학습·검증 표시는 바꾸지 않았다.** 남겨 둔 구간이 안정 상태나 고장 없음으로 확인되었다는 뜻은 아니다. 출력 0은 실제 기록값이며 결측으로 바꾸지 않는다. 비슷한 온도·출력 왕복이 다른 구간에도 있어 제어 반응과 고장 여부를 구분할 근거가 더 필요하다.

후반의 `TC-Z9`·냉각 온도는 반복 파형과 운전 단계의 영향을 나누어 살펴야 한다. 058의 08:14와 114의 20:10은 가열 온도의 참고 경계이며 이 세 센서의 확정 변화 시작이 아니다. 아래 값은 원자료와 센서별 유효 1분 평균을 다시 계산했다.

- **LOT 240058 · 냉각 온도와 `TC-Z9`는 구분해서 해석** — 검토 범위는 06/09 08:14:00~09:02:53이다. `TC-OUT1`은 봉우리 5개·골 5개, `TC-OUT2`는 봉우리 4개·골 5개가 확인된다. 봉우리끼리·골끼리의 간격은 각각 41~43분, 41~44분이다. **두 냉각 센서 모두 후반 봉우리·골이 앞선 같은 종류의 값들보다 낮아, 단순한 반복 위치 차이만은 아니다.**

| 센서·비교점 | 앞선 값 범위(℃) | 후반 시각·값(℃) | 직전 같은 비교점과 차이 |
| --- | --- | --- | --- |
| `TC-OUT1` 봉우리 | 앞선 4개 290.708~291.085 | 08:50 · 280.893 | −9.893℃ |
| `TC-OUT1` 골 | 앞선 4개 264.019~264.156 | 08:34 · 259.235 | −4.919℃ |
| `TC-OUT2` 봉우리 | 앞선 3개 283.388~283.712 | 08:21 · 280.227 | −3.271℃ |
| `TC-OUT2` 골 | 앞선 4개 265.436~266.482 | 08:45 · 257.636 | −8.846℃ |

`TC-Z9`의 317.441℃는 **05:26의 첫 관측값**이라 골인지 알 수 없다. 06:38 큰 봉우리 339.865℃ → 08:40 골 309.978℃ → 08:51 작은 반등 310.808℃만 보여, 첫 값과 뒤쪽 골이 같은 반복 위치인지 확인할 수 없다. 냉각의 후반 변화도 한 번뿐이고 같은 DP590의 다른 LOT가 없어 원인·유지 기간·새 보류 경계를 정하지 않는다. 08:14는 세 센서의 확정 변화 시작이 아니다. [058 상세 비교와 재계산](07_lot_anomaly_review_namelesspark/06_LOT_240058.ipynb)에서 파형을 함께 확인한다.

- **LOT 240058 · 건조 구역** — 06/09 08:40:00~09:02:53, `TC-Z1` · `TC-Z2` · `OP-Z1` · `OP-Z2`. 원자료 1,328행에서 네 센서 결측은 없고 마지막 09:02분에도 53개가 있다. 유효 1분 평균 08:50→09:02는 `OP-Z1` 63.582→1.938%, `OP-Z2` 10.530→0.520%, `TC-Z1` 97.734→96.160℃, `TC-Z2` 98.424→95.532℃다. 이 12분 동안 두 출력과 `TC-Z2`는 매분 낮아지지만 `TC-Z1`은 중간 반등이 있다. `OP-Z1`도 08:47~08:50에는 먼저 반등하므로 후반 전체를 단조 감소로 해석하지 않는다. 마지막 원자료의 두 출력은 0이나 각 1개 관측뿐이다. **08:40은 선택한 살펴볼 범위이지 현상 시작 시각이 아니다.** 같은 DP590의 다른 LOT가 없고 건조 구역의 별도 보류 기준도 정하지 않아 유지한다. 고장이나 건조 중단을 확정한 것은 아니다.

- **LOT 240061** — 06/15 03:12:03~03:26:59, `OP-Z3` · `TC-Z3`. 연이어 기록된 OP-Z3=0: 03:12:03~03:18:39, 389개 관측. TC-Z3 유효 1분 평균은 03:15 870.570℃에서 03:26 850.742℃로 변화. 가열 구역을 다시 남기기 시작한 02:31 이후에 발생.

- **LOT 240069** — 07/08 04:29:20~05:05:59, `OP-Z3` · `TC-Z3`. 연이어 기록된 OP-Z3=0: 04:50:34~04:55:50, 306개 관측. TC-Z3 유효 1분 평균은 04:53 870.496℃에서 05:05 851.993℃로 변화. 04:29:20 이전 관측 최고 94.2336%, 이후 검토 구간 최고 98.7936%.

- **LOT 240091** — 09/09 07:07:14~07:28:56, `OP-Z3` · `TC-Z3`. 연이어 기록된 OP-Z3=0: 07:07:14~07:12:42, 324개 관측. TC-Z3 유효 1분 평균은 07:10 870.509℃에서 07:26 851.784℃로 변화. 이후 97.3164% 평탄 관측 07:22:40~07:28:56.

- **LOT 240114** — 11/04 20:10:00~20:59:02, `TC-Z9` · `TC-OUT1` · `TC-OUT2`. 유효 1분 평균 TC-OUT1 봉우리: 20:09 296.912℃ → 20:50 285.875℃. TC-OUT2 골: 20:03 273.276℃ → 20:44 265.362℃. TC-Z9의 두 봉우리(16:47·20:03)에서 각각 55분 뒤는 330.204/313.914℃.

- **LOT 240114 · 건조 구역** — 11/04 20:30:00~20:59:02, `TC-Z1` · `TC-Z2` · `OP-Z1` · `OP-Z2`. 검증 자료의 관찰이며 기본안의 네 센서 표시는 0으로 유지한다. 원자료 1,707행에서 네 센서 결측은 없다. 마지막 20:59분은 3개뿐이라 평균을 쓰지 않고, 60개가 관측된 마지막 유효분 20:58과 비교한다. 1분 평균 20:46→20:58은 `OP-Z1` 78.892→2.535%, `OP-Z2` 20.574→0.877%로 12분 연속 감소한다. 반면 `TC-Z1`은 95.656→96.465℃로 오르고, `TC-Z2`는 96.462→95.786℃지만 20:52의 95.474℃ 이후 반등한다. `OP-Z1`도 20:37~20:46에는 먼저 반등하므로 모든 온도·출력이 후반 내내 함께 내려간 것은 아니다. 마지막 원자료의 두 출력은 0이나 각 1개 관측뿐이다. **20:30은 선택한 살펴볼 범위이지 현상 시작 시각이 아니다.** 고장·건조 중단을 확정하지 않으며 검증 자료로 임계값이나 보류 규칙을 조정하지 않는다.

출력의 높은 평탄값도 실제 설정 상한이라고 확정하지 않는다. 058의 TC-Z9 첫 관측값과 뒤쪽 최저값 비교, 114의 봉우리에서 같은 55분 경과 비교는 물리 조건이 같다는 뜻이 아니다. 자세한 근거는 `05_review_windows.csv`에 남겼다.

## 6. 표시를 붙인 뒤 강종·센서별로 남은 자료를 확인한다

아래의 ‘유효분’은 **그 센서에 관측이 30개 이상 있는 분**이다. ‘남은분’은 여기서 기본안의 센서별 표시를 적용한 결과다. 여러 센서가 같은 분에 보류되어도 센서별로 따로 센다.

| 구간·대상 | 표시한 시간 칸 | 관측 수 조건을 만족한 실제 유효 보류분 |
| --- | --- | --- |
| 240024 `CP-Z4`, 04:59~05:00 | 2칸 | 2분 |
| 240061 가열 8개, 01:04~02:30 | 87칸 | 센서당 87분, `OP-Z3`는 84분 |
| 240061 과시효 4개, 01:04~02:14 | 71칸 | 센서당 71분 |
| 240114 가열 8개, 20:10~20:59 | 50칸 | 센서당 49분 |
| 240114 과시효 4개, 20:19~20:59 | 41칸 | 센서당 40분 |

240061의 `OP-Z3`는 표시 범위 안의 3분에 센서 관측 수가 부족하다. 240114의 마지막 20:59분은 원자료가 3개뿐이어서 평균을 비워 두므로, 표시한 시간 칸보다 유효 보류분이 하나 적다. 시간 범위가 있다는 사실과 모델에 쓸 값이 있다는 사실을 구분한다.

‘기존 06 입력조건까지 확인한 분’에는 같은 구역의 다른 센서, 자기 센서의 1분·5분 전 값도 있어야 한다. **이 수치는 기존 06처럼 대상 센서의 표시만 적용한 경우**다. 입력 센서나 과거값의 표시를 전달하는 정책을 추가하면 다시 달라진다. 이 노트북은 새 모델을 학습하지 않는다.

In [ ]:
def interval_mask(frame, start, end):
    # 1분 칸과 원자료 구간이 조금이라도 겹치면 그 분에 표시합니다.
    return frame.MEAS_DT.le(end) & (frame.MEAS_DT + pd.Timedelta(minutes=1)).gt(start)

flags = pd.DataFrame(False, index=minute.index, columns=CHANNELS)
for _, window in applied.iterrows():
    hit = minute.LOT_NO.eq(window.LOT) & interval_mask(minute, window["시작"], window["끝"])
    sensors = window["채널"].split(";")
    flags.loc[hit, sensors] = True

remaining_rows = []
for _, sensors in GROUPS.items():
    for target in sensors:
        valid_value = minute[target].notna()
        kept = valid_value & ~flags[target]
        lag1 = minute.groupby("LOT_NO")[target].shift(1)
        lag5 = minute.groupby("LOT_NO")[target].shift(5)
        usable = kept & lag1.notna() & lag5.notna()
        others = [sensor for sensor in sensors if sensor != target]
        if others:
            usable &= minute[others].notna().all(axis=1)
        for (split, grade), group in minute.groupby(["SPLIT", "GRADE"], sort=True):
            index = group.index
            remaining_rows.append({
                "분할": split,
                "강종": grade,
                "센서": target,
                "LOT수": group.LOT_NO.nunique(),
                "유효분": int(valid_value.loc[index].sum()),
                "기본안보류분": int((valid_value & flags[target]).loc[index].sum()),
                "남은분": int(kept.loc[index].sum()),
                "기존06입력조건까지확인한분": int(usable.loc[index].sum()),
            })
remaining = pd.DataFrame(remaining_rows)
remaining["자료용도"] = remaining["분할"].map(SPLIT_NAME)
remaining = remaining.sort_values(["분할", "강종", "센서"]).reset_index(drop=True)
display(remaining.loc[remaining["분할"].eq("train")].pivot(index="센서", columns="강종", values="남은분"))
display(remaining.loc[remaining["기본안보류분"].gt(0)])

lot_rows = []
for lot, group in minute.groupby("LOT_NO", sort=True):
    valid_values = group[CHANNELS].notna()
    held = flags.loc[group.index] & valid_values
    lot_rows.append({
        "LOT": lot,
        "강종": lots.loc[lot, "GRADE"],
        "자료용도": SPLIT_NAME[SPLIT[lot]],
        "전체분": len(group),
        "유효센서분": int(valid_values.to_numpy().sum()),
        "보류센서분": int(held.to_numpy().sum()),
        "한센서이상보류된분": int(held.any(axis=1).sum()),
    })
lot_summary = pd.DataFrame(lot_rows)
display(lot_summary)
save_csv(remaining, "grade_channel_remaining")
save_csv(lot_summary, "exclusion_by_lot")
save_csv(minute, "minute_features")
save_csv(counts, "minute_sensor_counts")
save_csv(lots.reset_index(), "lot_grade_split")
flag_output = pd.concat([minute[["LOT_NO", "MEAS_DT", "GRADE", "SPLIT"]], flags], axis=1)
save_csv(flag_output, "target_flags")

## 7. 대상 센서·입력 센서·과거값은 서로 다르다

예를 들어 CP 급변이 10:00에 표시되었다면 **CP 예측 모델의 10:00 학습 행**은 보류된다. 그러나 `TC-Z4` 모델이 그 CP 값을 입력으로 쓰거나, CP 모델이 10:01·10:05에 그 값을 과거값으로 쓰는 문제는 별도다.

이전 참고 실험은 **대상 센서의 표시만 확인한다.** CP 표시가 다른 모델의 입력이나 1분·5분 전 값까지 자동으로 전달되지는 않는다. 따라서 ‘CP가 모든 모델에서 제거되었다’고 설명하면 안 된다.

후속 모델에서 선택할 일은 두 가지다. ① 센서별 관측 상태를 입력으로 허용할지, ② 입력·과거값의 원자료가 보류 표시된 경우 그 학습 행도 보류할지. 두 번째를 선택한다면 **원자료가 관측된 시각의 표시를 과거값과 같이 이동**해야 한다. 아래는 전달 원리를 보여주는 예이며 새 모델 정책을 실행한 결과가 아니다. 시험의 제외 표시는 계속 0으로 유지한다.

In [ ]:
cp_held = flags["CP-Z4"]
source_held_1m = cp_held.groupby(minute.LOT_NO).shift(1, fill_value=False)
source_held_5m = cp_held.groupby(minute.LOT_NO).shift(5, fill_value=False)
first_cp = applied.loc[applied["채널"].eq("CP-Z4")].iloc[0]
example_hit = minute.LOT_NO.eq(first_cp.LOT)
example_hit &= minute.MEAS_DT.between(
    first_cp["시작"].floor("min") - pd.Timedelta(minutes=1),
    first_cp["끝"].floor("min") + pd.Timedelta(minutes=6),
)
example = minute.loc[example_hit, ["LOT_NO", "MEAS_DT", "CP-Z4"]].copy()
example["CP예측대상보류"] = cp_held.loc[example_hit]
example["TC4자체보류"] = flags.loc[example_hit, "TC-Z4"]
example["CP1분전원자료에표시"] = source_held_1m.loc[example_hit]
example["CP5분전원자료에표시"] = source_held_5m.loc[example_hit]
display(example)

### 기존 06 입력 구성의 참고 계산: 표시 범위별 학습 행수

**기존 06 실험의 입력 구성을 그대로 가정한 참고 계산이다. 팀 모델 설계가 확정됐다는 뜻이 아니며, 모델을 학습하지 않는다. 시험 자료는 사용하지 않고 학습 9개 LOT만 센다.**

세 열 모두 같은 구역의 다른 센서 현재값·강종·LOT 경과시간·대상 센서의 1분 전과 5분 전 값이 있는 행에서 시작한다. **값이 존재하는지 확인하는 조건은 같고, 보류 표시를 확인하는 범위만 넓힌다.** 대상 센서의 표시만 확인한 뒤, 현재 입력 센서의 표시도 확인하고, 마지막에는 과거값이 관측된 시각의 표시까지 확인한다. 새로운 모델 방식이나 실제 적용 정책을 정한 것이 아니다.

한 행은 한 센서를 예측하는 1분 자료다. 표의 행수를 센서끼리 더하면 같은 시간대가 중복된다. 아래 비교를 계산해도 기존 적용 목록과 실제 표시는 바뀌지 않는다.

In [ ]:
# 기존 06의 입력 구성으로 행수만 비교합니다. flags 자체는 바꾸지 않습니다.
training = minute.loc[minute.LOT_NO.isin(TRAIN)].copy()
training_flags = flags.loc[training.index]
scopes = [("전체", training.index)]
for grade, group in training.groupby("GRADE", sort=True):
    scopes.append((grade, group.index))

comparison_rows = []
for zone, sensors in GROUPS.items():
    for target in sensors:
        others = [sensor for sensor in sensors if sensor != target]
        lag1 = training.groupby("LOT_NO")[target].shift(1)
        lag5 = training.groupby("LOT_NO")[target].shift(5)
        complete = training.QUALITY_OK & training[target].notna()
        complete &= lag1.notna() & lag5.notna()
        complete &= training.GRADE.notna() & training.ELAPSED_MIN.notna()
        if others:
            complete &= training[others].notna().all(axis=1)

        target_only = complete & ~training_flags[target]
        current_inputs = target_only.copy()
        if others:
            current_inputs &= ~training_flags[others].any(axis=1)
        past1 = training_flags[target].groupby(training.LOT_NO).shift(1, fill_value=False)
        past5 = training_flags[target].groupby(training.LOT_NO).shift(5, fill_value=False)
        past_inputs = current_inputs & ~past1 & ~past5

        for grade, index in scopes:
            comparison_rows.append({
                "구역": zone,
                "강종": grade,
                "센서": target,
                "대상만확인": int(target_only.loc[index].sum()),
                "현재입력까지확인": int(current_inputs.loc[index].sum()),
                "과거입력까지확인": int(past_inputs.loc[index].sum()),
            })
input_comparison = pd.DataFrame(comparison_rows)
save_csv(input_comparison, "input_flag_policy_comparison")

example_scopes = [
    ("전체", "CP-Z4"),
    ("전체", "TC-Z4"),
    ("전체", "TC-Z9"),
    ("DP590", "TC-Z4"),
    ("SPCEN", "OP-Z4"),
]
example_rows = []
for grade, target in example_scopes:
    chosen = input_comparison["강종"].eq(grade) & input_comparison["센서"].eq(target)
    example_rows.append(input_comparison.loc[chosen])
display(pd.concat(example_rows, ignore_index=True))

#### 현재 자료의 참고 결과: 모델 학습 없이 행수만 비교

| 학습 자료 범위 | 대상 센서 | 대상만 확인 | 현재 입력까지 확인 | 과거 입력까지 확인 |
| --- | --- | --- | --- | --- |
| 전체 | `CP-Z4` | 3,775 | 3,693 | 3,654 |
| 전체 | `TC-Z4` | 3,767 | 3,693 | 3,688 |
| 전체 | `TC-Z9` | 3,898 | 3,790 | 3,790 |
| DP590 | `TC-Z4` | 163 | 122 | 122 |
| SPCEN | `OP-Z4` | 120 | 118 | 118 |

DP590의 가열 8개 센서는 각각 163행에서 122행으로 25.2% 줄어든다. `TC-Z9` 자체에 보류 표시가 없어도 같은 구역의 입력 센서 표시를 확인하면 행수가 줄 수 있다. SPCEN의 `OP-Z4`도 현재 입력의 CP 표시 때문에 2행이 줄어든다.

DP590과 SPCEN은 각각 학습 LOT가 하나다. 남은 122행·118행은 한 LOT에서 시간상 이어진 분 자료이며 서로 독립적인 생산 조건 122개·118개를 뜻하지 않는다. 입력 표시 전달 여부를 비교할 때는 행수뿐 아니라 어떤 운전 조건이 남는지, 다른 LOT에서도 같은 관계가 보이는지를 함께 확인해야 한다.

이 집계는 **학습에 있는 6개 강종만** 대상으로 한다. 과거 입력까지 확인한 뒤 0행이 되는 강종·대상 센서 조합은 0개지만, 남은 자료가 충분하다는 보증은 아니다.

**SPCD는 시험 LOT 240117에만 있어 학습 자료가 없으며, 이 집계에 포함되지 않는다.**

입력 표시를 전달할지 결정할 때 행수 감소와 센서 관계를 함께 검토해야 한다. 전체 센서·강종별 값은 `05_input_flag_policy_comparison.csv`에 저장한다.

## 8. 그림으로 적용 범위와 경계를 다시 확인한다

붉은 음영은 **해당 축의 센서에 적용한 기본안의 보류 표시**다. 온도·출력·CP는 단위가 달라 축을 나눈다. CP 점은 원자료, 선은 1분 평균이다. 선택한 허용 편차가 물리적 정상 범위라는 뜻은 아니다.

In [ ]:
def finish_plot(fig, axes, filename, title):
    fig.suptitle(title, fontsize=13)
    for ax in np.asarray(axes).reshape(-1):
        ax.grid(alpha=0.2)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
        ax.legend(fontsize=8, loc="best")
    fig.savefig(FIGURES / filename, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def shade_applied(ax, lot, sensors):
    for _, window in applied.loc[applied.LOT.eq(lot)].iterrows():
        if set(sensors).intersection(window["채널"].split(";")):
            start = window["시작"].floor("min")
            end = window["끝"].floor("min") + pd.Timedelta(minutes=1)
            ax.axvspan(start, end, color="#d9534f", alpha=0.16)

for lot in [240058, 240061, 240114]:
    group = lot_minutes(lot)
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, layout="constrained")
    for sensor in HEAT_TC:
        axes[0].plot(group.index, group[sensor], linewidth=0.9, label=sensor)
    for sensor in HEAT_OP:
        axes[1].plot(group.index, group[sensor], linewidth=0.9, label=sensor)
    axes[2].plot(group.index, group["CP-Z4"], color="#4d5794", label="CP-Z4")
    axes[0].set_ylabel("온도 (℃, 가정)")
    axes[1].set_ylabel("제어출력 (%, 가정)")
    axes[2].set_ylabel("CP (단위 미확인)")
    for ax, sensors in zip(axes, [HEAT_TC, HEAT_OP, ["CP-Z4"]]):
        shade_applied(ax, lot, sensors)
    if lot == 240061:
        axes[0].plot(group.index, group["HEAT_TC_MEAN"], color="#222222", linewidth=1.5, label="가열 온도 4개 평균")
        axes[0].axhspan(startup_reference - 2, startup_reference + 2, color="#69b3a2", alpha=0.2, label="4개 평균의 사후 기준 ±2℃")
        axes[0].axvline(stable_start, color="#33836b", linestyle="--", label=f"안정구간 시작 {stable_start:%H:%M}")
    if lot == 240058:
        axes[2].axvline(cp_decline_start, color="#33836b", linestyle="--", label="분평균 지속 조건 07:43")
    title = f"LOT {lot} | 붉은 구간은 해당 축의 센서만 기본안에서 보류"
    finish_plot(fig, axes, f"phase_{lot}.png", title)

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex="row", layout="constrained")
for row, lot in enumerate([240058, 240061, 240114]):
    group = lot_minutes(lot)
    for sensor in AGING_TC:
        axes[row, 0].plot(group.index, group[sensor], linewidth=1, label=sensor)
    for sensor in AGING_OP:
        axes[row, 1].plot(group.index, group[sensor], linewidth=1, label=sensor)
    shade_applied(axes[row, 0], lot, AGING_TC)
    shade_applied(axes[row, 1], lot, AGING_OP)
    axes[row, 0].set_title(f"{lot} 과시효 온도", fontsize=10)
    axes[row, 1].set_title(f"{lot} 과시효 출력", fontsize=10)
    axes[row, 0].set_ylabel("온도 (℃, 가정)")
    axes[row, 1].set_ylabel("출력 (%, 가정)")
finish_plot(fig, axes, "aging_phase.png", "과시효 온도·출력도 구역 자체의 변화 시각으로 분리 | TC-Z9는 별도 검토")

cp_plot_lots = [240024, 240037, 240038, 240069, 240108, 240117]
fig, axes = plt.subplots(3, 2, figsize=(14, 10), layout="constrained")
for ax, lot in zip(axes.flat, cp_plot_lots):
    source = raw.loc[raw.LOT_NO.eq(lot)]
    group = lot_minutes(lot)
    ax.plot(source.MEAS_DT, source["CP-Z4"], ".", markersize=1.5, alpha=0.4, color="#64748b", label="원자료")
    ax.plot(group.index, group["CP-Z4"], linewidth=0.9, color="#3858a5", label="1분 평균")
    ax.axhspan(CP_CENTER - CP_HOLD_GAP, CP_CENTER + CP_HOLD_GAP, color="#69b3a2", alpha=0.12, label="선택한 ±0.05")
    shade_applied(ax, lot, ["CP-Z4"])
    ax.set_title(f"{lot} · {SPLIT_NAME[SPLIT[lot]]}", fontsize=10)
    ax.set_ylabel("CP (단위 미확인)")
finish_plot(fig, axes, "cp_both_directions.png", "양·음 방향에 같은 CP 기준 적용 | 시험은 표시만 확인")

fig, axes = plt.subplots(2, 1, figsize=(12, 6), layout="constrained")
for lot, label in [(240038, "240038 전체"), (240114, "240114 하강 전")]:
    group = lot_minutes(lot)
    if lot == 240114:
        group = group.loc[group.index < cooling_starts[(lot, "가열·균열")]]
    elapsed = (group.index - lots.loc[lot, "START"]).total_seconds() / 60
    axes[0].plot(elapsed, group["OP-Z4"], linewidth=1, label=label)
    axes[1].plot(elapsed, group["TC-Z4"], linewidth=1, label=label)
axes[0].set_ylabel("OP-Z4 (%, 가정)")
axes[1].set_ylabel("TC-Z4 (℃, 가정)")
axes[1].set_xlabel("LOT 시작 후 경과 분")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
fig.suptitle("같은 SPCEN의 출력 차이 | 240038 강제 제외를 철회하고 원인 검토", fontsize=13)
fig.savefig(FIGURES / "spcen_review.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 3.5), layout="constrained")
group = lot_minutes(240069)
ax.plot(group.index, group["CLN-IN"], color="#427c79", label="CLN-IN")
ax.set_ylabel("온도 (℃, 가정)")
finish_plot(fig, [ax], "cln_review.png", "240069 세정조 온도 | 초기 전체를 강제 제외하지 않고 검토")

### 현재 자료에서 그린 적용 범위

실행하지 않아도 다음 그림으로 기준의 위치를 확인할 수 있다. 붉은 음영은 대상 센서의 **1분 칸 전체**에 적용한 표시이며 원자료 삭제가 아니다. 관측 수가 부족한 평균은 선에서 빠진다.

**240058: CP 이탈과 가열 온도 감소의 경계를 분리한다.**

![240058 가열 온도·출력과 CP](../figures/05_exclusion_windows_namelesspark/phase_240058.png)

**240061: 가열 온도 4개 평균의 비교 수준과 초기 보류 구간.**

![240061 초기 온도 상승 구간](../figures/05_exclusion_windows_namelesspark/phase_240061.png)

**240114: 후반 가열 온도·출력은 변하지만 CP는 별도로 유지한다.**

![240114 후반 운전 단계](../figures/05_exclusion_windows_namelesspark/phase_240114.png)

**과시효 구역: 자체 온도 기준으로 시각을 정한다. 출력은 같은 순간부터 단조 감소하지 않는다.**

![과시효 온도와 출력의 별도 경계](../figures/05_exclusion_windows_namelesspark/aging_phase.png)

**CP: 양·음 방향에 같은 기준을 적용한다. 시험에서는 추가 제외하지 않는다.**

![양방향 CP 이탈 확인](../figures/05_exclusion_windows_namelesspark/cp_both_directions.png)

**같은 SPCEN: 관측 온도와 출력 차이는 확인되지만 부하·설정·고장 원인은 구분하지 못한다.**

![SPCEN의 출력과 온도 비교](../figures/05_exclusion_windows_namelesspark/spcen_review.png)

**240069: 세정조의 초기 낮은 온도와 상승은 검토 대상으로 유지한다.**

![240069 세정조 온도 검토](../figures/05_exclusion_windows_namelesspark/cln_review.png)

## 다음 모델 실험에 전달할 내용

**새 05를 실행해도 기존 06에는 자동 반영되지 않는다.** 기존 06은 `exclusion_windows.csv` 같은 옛 파일을 읽으며, 새 `data/processed/05_*.csv`를 읽지 않는다. 따라서 기존 06의 예측값·성능표는 **새 보류 기준으로 얻은 결과가 아니다.**

1. 기본안은 `05_applied_windows.csv`, 원인 미확인 검토는 `05_review_windows.csv`, 240038 비교 선택지는 `05_optional_comparison.csv`다. 원자료는 그대로다.
2. `05_target_flags.csv`에는 LOT·시각·강종·`SPLIT`(학습·검증·시험 구분)과 센서별 표시가 있다. `True`는 이번 기본안의 임시 보류 표시이며, `False`가 정상 확정을 뜻하지 않는다.
3. `05_minute_sensor_counts.csv`와 `05_minute_features.csv`에는 관측 수와 품질 처리 결과가 있다. 강종·센서별 남은 자료는 `05_grade_channel_remaining.csv`로 확인한다.
4. 240038의 강제 제외를 철회했으므로 **새 기본안에서는 SPCEN이 OP-Z4 학습에 남는다.** 이전 06의 ‘OP-Z4가 SPCEN을 학습하지 못했다’는 설명을 새 기준의 결과로 인용하면 안 된다.

학습 자료의 표시와 검증 자료의 표시는 용도를 나누어 연결해야 한다. **기존 06에서는** 학습 표시를 모델 학습에, 검증 표시를 이상 판단 임계값 계산에 사용했다. 이는 참고하는 이전 실험 방식이며 팀의 확정 설계가 아니다. 이 CSV는 검증 전체 평가나 사건 관찰에서 표시된 행을 자동으로 삭제하라는 표가 아니다.

후속 모델에서는 입력·과거값까지 표시를 전달할지 정하고, 파일 연결과 기존 사건 해석을 함께 수정한 뒤 다시 평가해야 한다. 선택값 비교는 학습·검증 자료에서 진행하고 시험 자료로 제외 기준이나 선택값을 조정하지 않는다. **시험 3개 LOT(240117·240121·240133)는 이미 사후 관찰했으므로, 아직 열어보지 않은 최종 평가 자료는 아니다.**

`TC`는 온도, `OP`는 제어출력이다. 단위 ℃·%는 교육용 데이터사전의 가정이고 CP 단위는 확인되지 않았다. 표시 밖의 구간도 고장 없음이 보증된 ‘정상 정답’이 아니라 학습에 사용하는 후보 자료다.
